# MSI5001 SER

Compare seven untuned models on AESDD: DummyClassifier, Logistic Regression, KNN, SVM,
Random Forest, MLP and a small CNN. The six scikit-learn classifiers retain their
default constructor settings (including MLP's 200-iteration limit).
CNN uses two convolution layers, Adam defaults, batch size 32 and 10 epochs.

Run the eight sections in order. Classical models use 40 MFCC mean/std features;
CNN uses 64×128 log-mel images. Evaluation reports accuracy and macro F1 for one
approximately 80/20 stratified group random split and LOSO by speaker.
Alternate takes stay together in the random split. LOSO scores pool the predictions
from all held-out speakers. Every model uses the same test recordings.

## 1. Install dependency & import library

Installation is optional when the selected kernel already has the dependencies.
Run installation before imports and restart the kernel after upgrades if required.
For local NVIDIA GPU support, install a CUDA-enabled PyTorch build using the
[official selector](https://pytorch.org/get-started/locally/) in your environment;
skip the installation cell once that environment is ready.

In [ ]:
%pip install --upgrade numpy pandas scikit-learn librosa soundfile gdown matplotlib ipython
%pip install torch

In [ ]:
from pathlib import Path
import re, zipfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Audio, Markdown
import librosa
import soundfile as sf
import gdown
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

## 2. Load dataset

Choose `local` or `drive`. Local input accepts a WAV directory, ZIP, or folder of ZIPs.
Drive input copies a public folder or dataset ZIP locally without mounting Drive.
The Drive sharing setting must be **Anyone with the link → Viewer**.
This section lists candidate AESDD files; audio validation and cleanup happen in section 3.

In [ ]:
SOURCE = 'local'                         # 'local' or 'drive'
LOCAL_PATH = r'./AESDD'                   # WAV directory, ZIP, or folder of ZIPs
DRIVE_URL = 'PASTE_PUBLIC_FILE_OR_FOLDER_URL_HERE'

root = Path(LOCAL_PATH).expanduser().resolve()
if SOURCE == 'drive':
    root = Path('./downloaded_AESDD').resolve()
    root.mkdir(exist_ok=True)
    if '/folders/' in DRIVE_URL:
        downloaded = gdown.download_folder(DRIVE_URL, output=str(root), use_cookies=False)
    else:
        downloaded = gdown.download(DRIVE_URL, output=str(root/'dataset.zip'), use_cookies=False)
    if not downloaded:
        raise RuntimeError('Download failed. Use a public anyone-with-the-link Drive URL.')
elif SOURCE != 'local':
    raise ValueError("SOURCE must be 'local' or 'drive'.")
if not root.exists():
    raise FileNotFoundError(f'Edit LOCAL_PATH: {root}')

def unzip(path):
    destination = path.with_suffix('')
    if not (destination/'.extracted').exists():
        with zipfile.ZipFile(path) as archive:
            if any(not (destination/name).resolve().is_relative_to(destination.resolve())
                   for name in archive.namelist()):
                raise ValueError('Unsafe archive path')
            archive.extractall(destination)
        (destination/'.extracted').touch()
    return destination

if root.is_file():
    root = unzip(root)
for _ in range(3):                       # Also handles ZIPs inside a downloaded folder
    for archive in list(root.rglob('*.zip')):
        unzip(archive)

codes = {'a':'Anger', 'd':'Disgust', 'f':'Fear', 'h':'Happiness', 's':'Sadness'}
pattern = re.compile(r'([adfhs])\s*[_ -]?(\d{1,2})\s*[_ -]?\(\s*(\d+)\s*\)\s*[a-z]*', re.I)
records = []
for path in sorted(root.rglob('*.wav')):
    match = pattern.fullmatch(path.stem)
    if not match or '__MACOSX' in path.parts:
        continue
    emotion, utterance, speaker = match.groups()
    emotion, speaker = emotion.lower(), str(int(speaker))
    records.append({'path':path, 'emotion':codes[emotion], 'speaker':speaker,
                    'group':f'{speaker}_{emotion}_{int(utterance)}'})
if not records:
    raise ValueError('No AESDD WAV filenames found. Check the dataset path and filenames.')
recordings = pd.DataFrame(records)
display(pd.DataFrame([{'Source':SOURCE, 'Dataset folder':str(root),
                       'Candidate recordings':len(recordings)}]))

## 3. Explore and clean dataset

Inspect the candidate counts, then remove byte-identical duplicates and skip
damaged, empty, non-finite or silent WAVs. Silence is checked on the mono signal;
the original samples are cached for preprocessing and playback.
Tables show counts before and after cleanup, removal reasons, speaker/emotion
distribution, and the first five usable recordings. Audio players preview those
five recordings. Playback normalization does not alter the model inputs.

In [ ]:
before_counts = recordings.emotion.value_counts().sort_index()
candidate_count = len(recordings)
clean_records, cleanup_log, audio_cache, seen = [], [], {}, set()
for record in recordings.to_dict('records'):
    path = record['path']
    try:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen:
            cleanup_log.append({'File':path.name, 'Status':'Duplicate',
                                'Reason':'Byte-identical to a retained recording'})
            continue
        audio, sr = sf.read(path, dtype='float32', always_2d=True)
        if not len(audio) or sr <= 0:
            raise ValueError('Empty recording or invalid sample rate')
        if not np.isfinite(audio).all():
            raise ValueError('Non-finite audio samples')
        if np.max(np.abs(audio.mean(axis=1))) < 1e-7:
            raise ValueError('Silent mono signal')
    except (RuntimeError, ValueError, OSError) as error:
        cleanup_log.append({'File':path.name, 'Status':'Invalid', 'Reason':str(error)})
        continue
    clean_records.append({**record, 'duration_s':len(audio)/sr,
                          'sample_rate':sr, 'channels':audio.shape[1]})
    audio_cache[path] = (audio, sr)
    seen.add(digest)
if not clean_records:
    raise ValueError('No usable recordings remain after cleanup.')
recordings = pd.DataFrame(clean_records)
cleanup_log = pd.DataFrame(cleanup_log, columns=['File', 'Status', 'Reason'])
cleanup_summary = pd.DataFrame({
    'Recordings':[candidate_count, (cleanup_log.Status == 'Duplicate').sum(),
                  (cleanup_log.Status == 'Invalid').sum(), len(recordings)]
}, index=['Before cleanup', 'Removed duplicates', 'Skipped invalid WAVs', 'After cleanup'])
display(Markdown('**Cleanup summary**'))
display(cleanup_summary)
if not cleanup_log.empty:
    display(Markdown('**Removed recordings**'))
    display(cleanup_log)

emotion_counts = pd.DataFrame({'Before cleanup':before_counts,
                               'After cleanup':recordings.emotion.value_counts()}).fillna(0).astype(int)
emotion_counts = emotion_counts.sort_index()
speaker_counts = pd.crosstab(recordings.speaker, recordings.emotion)
display(Markdown('**Recordings per emotion**'))
display(emotion_counts)
display(Markdown('**Usable recordings per speaker and emotion**'))
display(speaker_counts)
display(Markdown('**Recording duration (seconds)**'))
display(recordings.duration_s.describe().drop('count').to_frame('Seconds').round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
emotion_counts.plot.bar(ax=axes[0], color=['#94a3b8', '#2563eb'], rot=25)
axes[0].set(title='Emotion balance before and after cleanup', xlabel='Emotion', ylabel='Recordings')
axes[0].set_ylim(0, emotion_counts.to_numpy().max()*1.2)
axes[0].legend(loc='upper center', ncol=2, frameon=False)
recordings.duration_s.plot.hist(ax=axes[1], bins=20, color='#0d9488', edgecolor='white')
axes[1].set(title='Duration of usable recordings', xlabel='Duration (seconds)', ylabel='Recordings')
fig.tight_layout()
plt.show()

ax = speaker_counts.plot.bar(stacked=True, figsize=(9, 4), colormap='Set2', rot=0)
ax.set(title='Emotion distribution by speaker after cleanup', xlabel='Speaker', ylabel='Recordings')
ax.legend(title='Emotion', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.figure.tight_layout()
plt.show()

In [ ]:
first_five = recordings.head(5).copy()
first_five['file'] = first_five.path.map(lambda path: path.name)
display(Markdown('**First five usable recordings**'))
display(first_five[['file', 'emotion', 'speaker', 'duration_s', 'sample_rate', 'channels']]
        .reset_index(drop=True).round({'duration_s':2}))
for i, record in enumerate(first_five.itertuples(index=False), start=1):
    display(Markdown(f'**{i}. {record.path.name}** — {record.emotion}, speaker {record.speaker}'))
    audio, sr = audio_cache[record.path]
    display(Audio(data=audio.T, rate=sr))

## 4. Data preprocessing

Use the cleaned recordings from section 3. Convert to mono and resample to 16 kHz.
Extract 20 MFCC means and standard deviations for classical ML and 64×128 log-mel
images for CNN. These operations are independent per recording.
The example waveform and log-mel image illustrate the model inputs.
Learned scaling is deferred to section 6.

In [ ]:
features, images = [], []
for record in recordings.to_dict('records'):
    audio, sr = audio_cache[record['path']]
    audio = librosa.resample(audio.mean(axis=1), orig_sr=sr, target_sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=20)
    mel = librosa.feature.melspectrogram(y=audio, sr=16000, n_mels=64)
    mel = torch.tensor(librosa.power_to_db(mel), dtype=torch.float32)[None, None]
    image = nn.functional.interpolate(mel, size=(64,128), mode='bilinear', align_corners=False)[0].numpy()
    if not features:
        example_audio = audio
    features.append(np.r_[mfcc.mean(axis=1), mfcc.std(axis=1)])
    images.append(image)
metadata = recordings.copy()
X, Z = np.asarray(features), np.asarray(images)
encoder = LabelEncoder()
y = encoder.fit_transform(metadata.emotion)
display(pd.DataFrame([
    {'Input':'MFCC mean/std features', 'Shape':str(X.shape), 'Used by':'Six classical models'},
    {'Input':'Log-mel images', 'Shape':str(Z.shape), 'Used by':'CNN'},
    {'Input':'Emotion labels', 'Shape':str(y.shape), 'Used by':'All seven models'}
]))
display(pd.DataFrame({'Label':np.arange(len(encoder.classes_)), 'Emotion':encoder.classes_}))

In [ ]:
example = metadata.iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(np.arange(len(example_audio))/16000, example_audio, color='#2563eb', linewidth=0.7)
axes[0].set(title=f'Mono waveform: {example.path.name}', xlabel='Time (seconds)', ylabel='Amplitude')
image_plot = axes[1].imshow(Z[0, 0], origin='lower', aspect='auto', cmap='magma')
axes[1].set(title='CNN input: resized log-mel image', xlabel='Resized time bin', ylabel='Mel band')
fig.colorbar(image_plot, ax=axes[1], label='Power (dB)')
fig.tight_layout()
plt.show()

## 5. Train test split (random & LOSO)

Random: use the first fold of a shuffled five-fold stratified group split, keeping
alternate takes of the same speaker/emotion/utterance together.
LOSO: hold out one complete speaker at a time. LOSO follows speaker boundaries;
it does not stratify or move recordings between speakers.

In [ ]:
train, test = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
                   .split(X, y, groups=metadata.group))
splits = [('Random', train, test)]
splits += [('LOSO', tr, te) for tr, te in LeaveOneGroupOut().split(X, y, metadata.speaker)]
split_summary = pd.DataFrame([
    {'Split':('Random' if setting == 'Random' else 'LOSO: speaker '+metadata.speaker.iloc[test].iloc[0]),
     'Train':len(train), 'Test':len(test),
     'Test speakers':','.join(metadata.speaker.iloc[test].unique())}
    for setting, train, test in splits
])
split_summary.index = pd.RangeIndex(1, len(split_summary)+1, name='Fold')
display(split_summary)
random_train, random_test = splits[0][1:]
random_class_counts = pd.DataFrame({
    'Train':np.bincount(y[random_train], minlength=len(encoder.classes_)),
    'Test':np.bincount(y[random_test], minlength=len(encoder.classes_))
}, index=pd.Index(encoder.classes_, name='Emotion'))
display(Markdown('**Emotion counts in the random split**'))
display(random_class_counts)

## 6. Feature scaling

Fit a fresh StandardScaler on each fold's **training MFCC features** and use it to
transform that fold's training and test features. Normalize CNN images using one
global mean and standard deviation computed from that fold's **training images**.
Test recordings never contribute to fitted scaling statistics.

The table has one row per independently scaled split: one random split plus
one LOSO split per speaker. Training-image mean/std should be approximately 0/1
after normalization; test data is transformed using those same training statistics.

In [ ]:
scaled_splits = []
for split_name, (setting, train, test) in zip(split_summary.Split, splits):
    scaler = StandardScaler().fit(X[train])
    mean, std = Z[train].mean(), max(float(Z[train].std()), 1e-8)
    scaled_splits.append({
        'name':split_name, 'setting':setting, 'train':train, 'test':test,
        'X_train':scaler.transform(X[train]), 'X_test':scaler.transform(X[test]),
        'Z_train':(Z[train]-mean)/std, 'Z_test':(Z[test]-mean)/std,
        'y_train':y[train], 'y_test':y[test]
    })
display(pd.DataFrame([
    {'Split':fold['name'], 'Train clips':len(fold['train']), 'Test clips':len(fold['test']),
     'MFCC features':fold['X_train'].shape[1],
     'CNN train mean':float(fold['Z_train'].mean()),
     'CNN train std':float(fold['Z_train'].std())}
    for fold in scaled_splits
]).set_index('Split').round(4))

## 7. Model training

Train fresh instances of all seven models for every split. Only training arrays
are passed into fitting. CNN uses CUDA automatically when available; the six
scikit-learn models use CPU. No hyperparameter tuning or early stopping is used.
Convergence warnings remain visible for models using their default iteration limit.

In [ ]:
EPOCHS, BATCH_SIZE = 10, 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
display(pd.DataFrame([{
    'CNN device':str(DEVICE), 'GPU':torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'Not used',
    'PyTorch':torch.__version__, 'Epochs':EPOCHS, 'Batch size':BATCH_SIZE
}]))
models = {'DummyClassifier':DummyClassifier(), 'Logistic Regression':LogisticRegression(),
          'KNN':KNeighborsClassifier(), 'SVM':SVC(),
          'Random Forest':RandomForestClassifier(), 'MLP':MLPClassifier()}

def train_cnn(images, labels):
    torch.manual_seed(42)
    inputs = torch.tensor(images)
    targets = torch.tensor(labels, dtype=torch.long)
    loader = DataLoader(TensorDataset(inputs, targets), batch_size=BATCH_SIZE, shuffle=True)
    model = nn.Sequential(
        nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32,len(encoder.classes_))
    ).to(DEVICE)
    optimizer, loss_fn = torch.optim.Adam(model.parameters()), nn.CrossEntropyLoss()
    for _ in range(EPOCHS):
        model.train()
        for batch, labels in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(batch.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
    return model

trained_splits = []
for fold in scaled_splits:
    fitted = {}
    for name, estimator in models.items():
        np.random.seed(42)
        fitted[name] = clone(estimator).fit(fold['X_train'], fold['y_train'])
    fitted['CNN'] = train_cnn(fold['Z_train'], fold['y_train'])
    trained_splits.append(fitted)
display(pd.DataFrame([
    {'Split':fold['name'], 'Trained models':len(fitted), 'Status':'Complete'}
    for fold, fitted in zip(scaled_splits, trained_splits)
]))

## 8. Model prediction and evaluation

Predict on held-out test arrays after training. Report accuracy and macro F1 for
the random split and pooled LOSO predictions, using all five emotion labels.

Accuracy is displayed as a percentage; macro F1 is displayed from 0 to 1.
The two charts compare random-split and pooled LOSO performance for all models.

In [ ]:
predictions = []
for fold, fitted in zip(scaled_splits, trained_splits):
    for name, model in fitted.items():
        if name == 'CNN':
            model.eval()
            with torch.no_grad():
                predicted = np.concatenate([
                    model(torch.tensor(fold['Z_test'][i:i+64]).to(DEVICE))
                    .argmax(1).cpu().numpy() for i in range(0,len(fold['Z_test']),64)
                ])
        else:
            predicted = model.predict(fold['X_test'])
        predictions.append((fold['setting'], name, fold['y_test'], predicted))

results = []
for name in [*models, 'CNN']:
    row = {'Model':name}
    for setting in ['Random', 'LOSO']:
        selected = [p for p in predictions if p[0]==setting and p[1]==name]
        actual = np.concatenate([p[2] for p in selected])
        predicted = np.concatenate([p[3] for p in selected])
        row[f'{setting} accuracy'] = accuracy_score(actual, predicted)
        row[f'{setting} macro F1'] = f1_score(actual, predicted, average='macro',
                                            labels=np.arange(len(encoder.classes_)), zero_division=0)
    results.append(row)
summary = pd.DataFrame(results).set_index('Model')
display(summary.style.format({
    'Random accuracy':'{:.1%}', 'Random macro F1':'{:.3f}',
    'LOSO accuracy':'{:.1%}', 'LOSO macro F1':'{:.3f}'
}).set_caption('Seven-model comparison — accuracy and macro F1'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, metric, columns in zip(axes, ['Accuracy', 'Macro F1'],
        [['Random accuracy', 'LOSO accuracy'], ['Random macro F1', 'LOSO macro F1']]):
    values = summary[columns].copy()
    values.columns = ['Random', 'LOSO (pooled)']
    values.plot.barh(ax=ax, color=['#2563eb', '#f59e0b'])
    ax.set(title=metric, xlabel=metric, ylabel='', xlim=(0, 1))
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(loc='lower right')
axes[0].invert_yaxis()
axes[0].xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle('SER model performance: random split vs LOSO')
fig.tight_layout()
plt.show()

Package versions, initialization and device can change results.
Outputs are cleared in the repository: run the notebook to generate tables, charts
and audio players in VS Code or Colab. Embedded audio previews are only generated
when the playback cell is run.

References: [gdown](https://github.com/wkentaro/gdown),
[scikit-learn](https://scikit-learn.org/stable/supervised_learning.html),
[PyTorch installation](https://pytorch.org/get-started/locally/).